# variant-manifests.ipynb

Figure out which of our dataset has somatic & germline variants.  
## Requires:
- `manifests/germline_manifest_20250407_121700.csv`, file metadata of germline variant call files (.vcf) hosted on CAVATICA;
- `manifests/somatic_manifest_20250407_121455.csv`, ditto for somatic .vcf files on CAVATICA.

In [ ]:
import sevenbridges as sbg
import pandas as pd
import pathlib
import os
import shutil

# import data_imports.py
import sys
sys.path.append('../../src')
from data_imports import *

pd.set_option('display.max_columns', None)

In [ ]:
# Shared functions
def load_variant_file_manifest(path):
    df = pd.read_csv(path,index_col=0)
    df=df[df.name.str.endswith('.vcf.gz')]
    return df

In [ ]:
BIOSAMPLES = import_biosamples()
PATIENTS=import_patients()

In [ ]:
PATIENTS.head()

## How many patients with germline variant calls?
1839

In [ ]:
def summarize_germline_variants(patients=None):
    if patients is None:
        patients=import_patients()
    germline_variants_manifest_path = 'manifests/germline_manifest_20250407_121700.csv'
    df = load_variant_file_manifest(germline_variants_manifest_path)
    df = df[df['Kids First Participant ID'].isin(patients.index)]
    return len(set(df['Kids First Participant ID']))
summarize_germline_variants(PATIENTS)

## How many biosamples with somatic variant calls?
2354

In [ ]:
def summarize_somatic_variants(biosamples=None):
    if biosamples is None:
        biosamples=import_biosamples()
    somatic_variants_manifest_path = 'manifests/somatic_manifest_20250407_121455.csv'
    df = load_variant_file_manifest(somatic_variants_manifest_path)
    #return df
    df = df[df['Kids First Biospecimen ID'].isin(biosamples.index)]
    return len(set(df['Kids First Biospecimen ID']))
summarize_somatic_variants(BIOSAMPLES)

## Which projects do our biosamples belong to?
There are 7 folders in CAVATICA/datasets/PBTA-CBTN. Which do our samples belong to?
```
Overlap with CMI: 0
Overlap with DGD: 0
Overlap with GFAC_TGEN: 0
Overlap with MiOncoseq: 0
Overlap with OligoNation: 0
Overlap with Project12: 872
Overlap with X01: 1431
```

In [ ]:
def load_biosamples_from_manifest(filename):
    df = pd.read_csv(filename, sep='\t')
    biosamples = set(df['Kids First Biospecimen ID'])
    return biosamples

def check_biosample_membership():
    projects = [
        'CMI',
        'DGD',
        'GFAC_TGEN',
        'MiOncoseq',
        'OligoNation',
        'Project12',
        'X01'
    ]
    cohort_set = set(BIOSAMPLES.index)
    for project in projects:
        pathname = 'manifests/harmonized-data-CBTN-'+project+'-manifest.tsv'
        project_set = load_biosamples_from_manifest(pathname)
        print(f'Overlap with {project}: {len(cohort_set & project_set)}')

check_biosample_membership()

## How many of our samples have associated somatic variant calls, by project?
```
Simple somatic variant files in CBTN-Project12: 848
Simple somatic variant files in CBTN-X01: 1431
```

In [ ]:
def load_project_manifest(project):
    df = pd.read_csv(f'manifests/harmonized-data-CBTN-{project}-manifest.tsv',sep='\t')
    return df
def get_somatic_simple_variants(project):
    df = load_project_manifest(project)
    df = df[df.name.map(lambda x: x.endswith('consensus_somatic.norm.annot.public.vcf.gz'))]
    df = df[df['Kids First Biospecimen ID'].isin(BIOSAMPLES.index)]
    return df

In [ ]:
def summarize_somatic_variants_by_project():
    for project in ['Project12','X01']:
        df = get_somatic_simple_variants(project)
        print(f'Simple somatic variant files in project CBTN-{project}: {len(df)}')
summarize_somatic_variants_by_project()